In [15]:
import pandas as pd
import logging
import os
import psycopg2
import json
import requests
from supabase import create_client
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv(".env")

DB_HOST = os.getenv("aws-1-eu-central-1.pooler.supabase.com")
DB_PORT = os.getenv("5432")
DB_NAME = os.getenv("postgres")
DB_USER = os.getenv("postgres.yrrqxyumwzbocuijewsl")
DB_PASSWORD = os.getenv("Khalidtechky")

SUPABASE_URL = os.getenv("https://yrrqxyumwzbocuijewsl.supabase.co")
SUPABASE_KEY = os.getenv("sb_publishable_Su1KCmUaQI9iqRukOxkpgw_kYPmn1Ow")

In [16]:
df = pd.read_csv("sales_2026-04-06.csv")

In [17]:
df.head()

,order_id,product_name,category,quantity,unit_price_usd,order_date,customer_id,status
0,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed
1,1002,Ankara Dress,Fashion,3,45.0,15/01/2024,C002,completed
2,1003,Standing Fan,Home,2,78.5,15/01/2024,C003,pending
3,1004,iPhone 15 Case,Electronics,5,12.0,16/01/2024,C001,completed
4,1005,Leather Handbag,Fashion,1,95.0,16/01/2024,C004,cancelled


In [18]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Project started successfully")

2026-04-13 11:19:17,407 - INFO - Project started successfully


In [19]:
def extract_sales(filepath):
    logging.info(f"Extracting sales data from {filepath}")

    try:
        df = pd.read_csv(filepath)
        logging.info(f"Sales data extracted successfully. Shape: {df.shape}")
        print(df.head())  # REQUIRED for testing step-by-step
        return df

    except Exception as e:
        logging.error(f"Failed to read CSV file: {e}")
        return None

In [20]:
sales_df = extract_sales("sales_2026-04-06.csv")

2026-04-13 11:19:18,182 - INFO - Extracting sales data from sales_2026-04-06.csv
2026-04-13 11:19:18,190 - INFO - Sales data extracted successfully. Shape: (15, 8)


   order_id        product_name     category  quantity  unit_price_usd  \
0      1001  Samsung Galaxy A54  Electronics         1           320.0   
1      1002        Ankara Dress      Fashion         3            45.0   
2      1003        Standing Fan         Home         2            78.5   
3      1004      iPhone 15 Case  Electronics         5            12.0   
4      1005     Leather Handbag      Fashion         1            95.0   

   order_date customer_id     status  
0  15/01/2024        C001  completed  
1  15/01/2024        C002  completed  
2  15/01/2024        C003    pending  
3  16/01/2024        C001  completed  
4  16/01/2024        C004  cancelled  


In [21]:
def transform_sales(df):
    logging.info("Starting sales data transformation")

    try:
        # Convert date
        df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=True)
        logging.info("Converted order_date to datetime")

        #  Fix numeric columns
        df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
        df["unit_price_usd"] = pd.to_numeric(df["unit_price_usd"], errors="coerce")

        logging.info("Converted quantity and unit_price_usd to numeric")

        #  Create total value in USD
        df["total_value_usd"] = df["quantity"] * df["unit_price_usd"]
        logging.info("Created total_value_usd column")

        #  Check missing values
        missing = df.isnull().sum()
        logging.info(f"Missing values per column:\n{missing}")

        print(df.head())

        logging.info("Sales transformation completed successfully")

        return df

    except Exception as e:
        logging.error(f"Sales transformation failed: {e}")
        return None

In [22]:
sales_transformed = transform_sales(sales_df)

2026-04-13 11:19:18,932 - INFO - Starting sales data transformation
2026-04-13 11:19:18,940 - INFO - Converted order_date to datetime
2026-04-13 11:19:18,951 - INFO - Converted quantity and unit_price_usd to numeric
2026-04-13 11:19:18,957 - INFO - Created total_value_usd column
2026-04-13 11:19:18,965 - INFO - Missing values per column:
order_id           0
product_name       0
category           0
quantity           0
unit_price_usd     1
order_date         0
customer_id        1
status             0
total_value_usd    1
dtype: int64
2026-04-13 11:19:18,981 - INFO - Sales transformation completed successfully


   order_id        product_name     category  quantity  unit_price_usd  \
0      1001  Samsung Galaxy A54  Electronics         1           320.0   
1      1002        Ankara Dress      Fashion         3            45.0   
2      1003        Standing Fan         Home         2            78.5   
3      1004      iPhone 15 Case  Electronics         5            12.0   
4      1005     Leather Handbag      Fashion         1            95.0   

  order_date customer_id     status  total_value_usd  
0 2024-01-15        C001  completed            320.0  
1 2024-01-15        C002  completed            135.0  
2 2024-01-15        C003    pending            157.0  
3 2024-01-16        C001  completed             60.0  
4 2024-01-16        C004  cancelled             95.0  


In [23]:
supabase = create_client("https://yrrqxyumwzbocuijewsl.supabase.co", "sb_publishable_Su1KCmUaQI9iqRukOxkpgw_kYPmn1Ow")

logging.info("Supabase client created successfully")

2026-04-13 11:19:19,365 - INFO - Supabase client created successfully


In [24]:
def extract_customers():
    logging.info("Extracting customers data from Supabase")

    try:
        response = supabase.table("customers").select("*").execute()

        df_customers = pd.DataFrame(response.data)

        logging.info(f"Customers extracted successfully. Shape: {df_customers.shape}")

        print(df_customers.head())

        return df_customers

    except Exception as e:
        logging.error(f"Failed to extract customers: {e}")
        return None

In [25]:
customers_df = extract_customers()

2026-04-13 11:19:20,067 - INFO - Extracting customers data from Supabase
2026-04-13 11:19:21,177 - INFO - HTTP Request: GET https://yrrqxyumwzbocuijewsl.supabase.co/rest/v1/customers?select=%2A "HTTP/2 200 OK"
2026-04-13 11:19:21,189 - INFO - Customers extracted successfully. Shape: (15, 8)


   order_id        product_name     category  quantity  unit_price_usd  \
0      1001  Samsung Galaxy A54  Electronics         1           320.0   
1      1002        Ankara Dress      Fashion         3            45.0   
2      1003        Standing Fan         Home         2            78.5   
3      1004      iPhone 15 Case  Electronics         5            12.0   
4      1005     Leather Handbag      Fashion         1            95.0   

   order_date customer_id     status  
0  15/01/2024        C001  completed  
1  15/01/2024        C002  completed  
2  15/01/2024        C003    pending  
3  16/01/2024        C001  completed  
4  16/01/2024        C004  cancelled  


In [26]:
sales_df["order_date"] = pd.to_datetime(sales_df["order_date"], dayfirst=True)
sales_df["total_value_usd"] = sales_df["quantity"] * sales_df["unit_price_usd"]

In [46]:
customers_df.head()
sales_df.head()

,order_id,product_name,category,quantity,unit_price_usd,order_date,customer_id,status
0,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed
1,1002,Ankara Dress,Fashion,3,45.0,15/01/2024,C002,completed
2,1003,Standing Fan,Home,2,78.5,15/01/2024,C003,pending
3,1004,iPhone 15 Case,Electronics,5,12.0,16/01/2024,C001,completed
4,1005,Leather Handbag,Fashion,1,95.0,16/01/2024,C004,cancelled


In [47]:
def merge_data(sales_df, customers_df):
    logging.info("Merging datasets")

    merged_df = pd.merge(
        sales_df,
        customers_df,
        on="customer_id",
        how="left"
    )

    logging.info(f"Merged data shape: (merged_df.shape)")
    return merged_df

In [48]:
df_merged = merge_data(sales_df, customers_df)
df_merged.head()

2026-04-13 11:24:35,107 - INFO - Merging datasets
2026-04-13 11:24:35,492 - INFO - Merged data shape: (merged_df.shape)


,order_id_x,product_name_x,category_x,quantity_x,unit_price_usd_x,order_date_x,customer_id,status_x,order_id_y,product_name_y,category_y,quantity_y,unit_price_usd_y,order_date_y,status_y
0,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed,1001.0,Samsung Galaxy A54,Electronics,1.0,320.0,15/01/2024,completed
1,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed,1004.0,iPhone 15 Case,Electronics,5.0,12.0,16/01/2024,completed
2,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed,1013.0,Rice Cooker,Home,2.0,65.0,19/01/2024,pending
3,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed,1015.0,Samsung Galaxy A54,Electronics,1.0,320.0,15/01/2024,completed
4,1002,Ankara Dress,Fashion,3,45.0,15/01/2024,C002,completed,1002.0,Ankara Dress,Fashion,3.0,45.0,15/01/2024,completed


In [49]:
df_merged["total_value_usd"] = df_merged["quantity_x"] * df_merged["unit_price_usd_x"]

In [50]:
exchange_rate = 1500  # example (replace with API later)

df_merged["total_value_ngn"] = df_merged["total_value_usd"] * exchange_rate

In [51]:
df_final = df_merged.dropna()   #clean final dataset

In [31]:
pd.read_csv("sales_2026-04-06.csv")

,order_id,product_name,category,quantity,unit_price_usd,order_date,customer_id,status
0,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed
1,1002,Ankara Dress,Fashion,3,45.0,15/01/2024,C002,completed
2,1003,Standing Fan,Home,2,78.5,15/01/2024,C003,pending
3,1004,iPhone 15 Case,Electronics,5,12.0,16/01/2024,C001,completed
4,1005,Leather Handbag,Fashion,1,95.0,16/01/2024,C004,cancelled
5,1006,Microwave Oven,Home,1,150.0,16/01/2024,C002,completed
6,1007,Wireless Earbuds,Electronics,2,55.0,17/01/2024,C005,completed
7,1008,Ankara Dress,Fashion,2,45.0,17/01/2024,C003,completed
8,1009,Blender,Home,1,NaN,17/01/2024,C006,pending
9,1010,Phone Charger,Electronics,3,8.0,18/01/2024,NaN,completed


In [32]:
from datetime import datetime, timedelta

today = datetime.today()
D_MINUS_1 = today - timedelta(days=1)

date_str = D_MINUS_1.strftime('%Y-%m-%d')

date_str

'2026-04-12'

In [33]:
filename = f"sales_{"sales_2026-04-06"}.csv"

filename

'sales_sales_2026-04-06.csv'

In [34]:
def extract_sales(filename):
    logging.info(f"Reading file: {filename}")

    try:
        df = pd.read_csv(filename)
        logging.info(f"Sales data loaded: {df.shape}")
        return df

    except Exception as e:
        logging.error(f"Failed to load sales file: {e}")
        return None
    

In [35]:
sales_df = extract_sales("sales_2026-04-06.csv")
sales_df.head()

2026-04-13 11:19:27,520 - INFO - Reading file: sales_2026-04-06.csv
2026-04-13 11:19:27,525 - INFO - Sales data loaded: (15, 8)


,order_id,product_name,category,quantity,unit_price_usd,order_date,customer_id,status
0,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed
1,1002,Ankara Dress,Fashion,3,45.0,15/01/2024,C002,completed
2,1003,Standing Fan,Home,2,78.5,15/01/2024,C003,pending
3,1004,iPhone 15 Case,Electronics,5,12.0,16/01/2024,C001,completed
4,1005,Leather Handbag,Fashion,1,95.0,16/01/2024,C004,cancelled


In [36]:
def extract_customers():
    logging.info("Extracting customers from Supabase")

    try:
        response = supabase.table("customers").select("*").execute()
        df = pd.DataFrame(response.data)

        logging.info(f"Customers loaded: {df.shape}")
        return df

    except Exception as e:
        logging.error(f"Failed to extract customers: {e}")
        return None

In [37]:
customers_df = extract_customers()
customers_df.head()

2026-04-13 11:19:28,583 - INFO - Extracting customers from Supabase
2026-04-13 11:19:29,326 - INFO - HTTP Request: GET https://yrrqxyumwzbocuijewsl.supabase.co/rest/v1/customers?select=%2A "HTTP/2 200 OK"
2026-04-13 11:19:29,331 - INFO - Customers loaded: (15, 8)


,order_id,product_name,category,quantity,unit_price_usd,order_date,customer_id,status
0,1001,Samsung Galaxy A54,Electronics,1,320.0,15/01/2024,C001,completed
1,1002,Ankara Dress,Fashion,3,45.0,15/01/2024,C002,completed
2,1003,Standing Fan,Home,2,78.5,15/01/2024,C003,pending
3,1004,iPhone 15 Case,Electronics,5,12.0,16/01/2024,C001,completed
4,1005,Leather Handbag,Fashion,1,95.0,16/01/2024,C004,cancelled


In [38]:
url = "https://open.er-api.com/v6/latest/USD"

response = requests.get(url)
data = response.json()

ngn_rate = data["rates"]["NGN"]

print(ngn_rate)

1362.350542


In [39]:
def extract_exchange_rate():
    logging.info("Fetching live exchange rate from API")

    try:
        url = "https://open.er-api.com/v6/latest/USD"
        response = requests.get(url)
        data = response.json()

        rate = data["rates"]["NGN"]

        logging.info(f"Exchange rate fetched: 1 USD = {rate} NGN")

        return rate

    except Exception as e:
        logging.error(f"Failed to fetch exchange rate: {e}")
        return None

In [52]:
df_merged["total_value_ngn"] = df_merged["total_value_usd"] * exchange_rate

In [53]:
exchange_rate = extract_exchange_rate()
exchange_rate

2026-04-13 11:25:26,278 - INFO - Fetching live exchange rate from API
2026-04-13 11:25:29,264 - INFO - Exchange rate fetched: 1 USD = 1362.350542 NGN


1362.350542

In [54]:
df_merged["total_value_ngn"] = df_merged["total_value_usd"] * exchange_rate

In [60]:
sales_df = sales_df.drop_duplicates()   #drop duplicates

sales_df = sales_df.dropna(subset=["unit_price_usd"])    #remove missing unit price usd

sales_df = sales_df.dropna(subset=["customer_id"])    #remove missing customer id

sales_df.columns = sales_df.columns.str.strip().str.lower()
customers_df.columns = customers_df.columns.str.strip().str.lower()     #clean column names

df_merged = df_merged[df_merged["status_x"].str.lower() != "cancelled"]     #remove cancelled orders

df_merged["unit_price_ngn"] = df_merged["unit_price_usd_x"] * exchange_rate    #currency conversion

df_merged["total_value_ngn"] = df_merged["unit_price_ngn"] * df_merged["quantity_x"]    #total value ngn column

In [62]:
def validate_data(df):
    print(" Running data validation checks...")

    #  Check nulls
    if df.isnull().sum().sum() > 0:                  #ensures no missing values
        print(" WARNING: There are still null values in the dataset")
    else:
        print(" No null values found")

    #  quantity > 0
    if (df["quantity"] <= 0).any():     #ensures quantity is real with no 0 or negative sales
        print(" WARNING: Some quantity values are <= 0")
    else:
        print(" All quantity values are valid")

    #  unit_price_usd > 0          #ensures pricing is valid
    if (df["unit_price_usd"] <= 0).any():
        print(" WARNING: Some unit_price_usd values are <= 0")
    else:
        print(" All unit_price_usd values are valid")

    #  rows count: shows final row counting 
    print(f" Rows ready for loading: {len(df)}")

In [65]:
df_final = df_merged  #ensure dataframe is ready

In [66]:
df_final = df_final.where(pd.notnull(df_final), None)    #clean

In [137]:
def load_to_supabase(df, exchange_rate):
    import psycopg2
    try:
        logging.info("Connecting to Supabase..")

        conn = psycopg2.connect(
            host="aws-1-eu-central-1.pooler.supabase.com",
            database="postgres",  # Changed - to =
            user="postgres.yrrqxyumwzbocuijewsl", # Ensure this has =
            password="Khalidtechky", # Ensure this has =
            port=5432
        )
        
        cursor = conn.cursor() 
        
        insert_query = """
        INSERT INTO daily_sales_report (
            customer_id,
            quantity,
            unit_price_usd,
            total amount_usd,
            total_onount_ngn,
            order date
        )VALUES（（%，%，%，%，%，%）
        """

        
        for _, row in df.iterrows():
            cursor.execute(insert_query, (
                row["customer_id"],
                row["quantity"],
                row["unit_price_usd"],
                row["total_amount_usd"],
                row("total_amount_ngn"),
                row["order_date"]
            ))

            conn.commit()
            cursor.close()
            conn.close()


            print(f" Insert successful!")
            print(f" Rows inserted: {len(df_final)}")
            print(f" Exchange rate used: {exchange_rate}")

            logging.info("Data successfully loaded to supabase")

    except Exception as e:
        logging.error(f"Error loading to supabase: {e}")
            

In [149]:
# main pipeline
def main():
    logging. info("Pipeline started")
    #Extract
    sales_df= extract_sales(filename)
    customers_df = extract_customers?
    
    
    logging.info("Extraction step completed")
# Transform
    def transform_sales(sales_df, customers_df):
# Validate
        validate_data = validate_data(final_df)
#  Load
    load_to_supabase (validate_data, exchange_rate)
    logging.info("Pipeline finished successfully")

In [150]:
main()

2026-04-13 13:06:10,918 - INFO - Pipeline started
2026-04-13 13:06:10,920 - INFO - Reading file: sales_sales_2026-04-06.csv
2026-04-13 13:06:10,925 - ERROR - Failed to load sales file: [Errno 2] No such file or directory: 'sales_sales_2026-04-06.csv'
2026-04-13 13:06:10,928 - INFO - Extraction step completed
2026-04-13 13:06:10,931 - INFO - Connecting to Supabase..
2026-04-13 13:06:14,674 - ERROR - Error loading to supabase: 'function' object has no attribute 'iterrows'
2026-04-13 13:06:14,680 - INFO - Pipeline finished successfully


Signature: extract_customers()
Docstring: <no docstring>
File:      c:\users\user\appdata\local\temp\ipykernel_17404\1145909025.py
Type:      function